In [1]:



import os, json, warnings
warnings.filterwarnings("ignore")

from math import pi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, r2_score, mean_absolute_error)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     GridSearchCV)
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from statsmodels.stats.multicomp import pairwise_tukeyhsd

DATA_PATH = "/content/sample_data/Dataset_200_students (4).csv"
OUTDIR = "figs"
os.makedirs(OUTDIR, exist_ok=True)
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "serif"
RNG = 42

# Capture all printed output in addition to stdout
LOG_LINES = []
def log(msg=""):
    print(msg)
    LOG_LINES.append(str(msg))

# ============================================================================
# PART A — DESCRIPTIVE CLUSTERING
# ============================================================================

# ----------------------------------------------------------------------------
# Step 1. Load and clean the data
# ----------------------------------------------------------------------------
log("=" * 78)
log("STEP 1 — LOAD AND CLEAN THE DATA")
log("=" * 78)

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()                 # remove header whitespace
for c in df.select_dtypes("object").columns:        # strip cell whitespace
    df[c] = df[c].str.strip()

log(f"Loaded {len(df)} rows, {len(df.columns)} columns")
log(f"Columns: {list(df.columns)}")
log(f"Missing values:\n{df.isna().sum()[df.isna().sum() > 0]}")

# ----------------------------------------------------------------------------
# Step 2. Encode variables
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 2 — ENCODE VARIABLES (ordinal / cyclic / binary / one-hot)")
log("=" * 78)

# Outcome: CGPA bands -> band midpoints
df["CGPA"] = df["CGPA?"].map({"2.0-2.5": 2.25, "2.5-3.0": 2.75,
                              "3.0-3.5": 3.25, "3.5-4.0": 3.75})

# Pre-specified ordinal maps (avoid alphabetic LabelEncoder ordering)
df["study_hrs_n"] = df["study_hrs"].map(
    {"Less than 1 hour":1,"1-2 hours":2,"3-4 hours":3,"More than 4 hours":4})
df["sleep_hrs_n"] = df["sleep_hrs"].map(
    {"Less than 4 hours":1,"4-6 hours":2,"6-8 hours":3,"More than 8 hours":4})
df["screen_n"] = df["screen_time"].map(
    {"Less than 1 hour":1,"1-3 hours":2,"4-6 hours":3,"More than 6 hours":4})
df["travel_n"] = df["travel_time"].map(
    {"Less than 15 minutes":1,"15-30 minutes":2,"30-60 minutes":3,
     "More than 1 hour":4,"More than 2 hour":5})
df["stress_n"] = df["stress"].map(
    {"Never":1,"Rarely":2,"Sometimes":3,"Often":4,"Always":5})
df["fin_cond_n"] = df["fin_cond"].map(
    {"Low Income":1,"Middle Income":2,"High Income":3})
df["balance_n"] = df["balance_diff"].map({"No":1,"Maybe":2,"Yes":3})
df["work_hrs_n"] = df["work_hrs"].map(
    {"Less than 5 hours":4,"5-10 hours":7.5,"11-20 hours":15,
     "More than 20 hours":22}).fillna(0)

# Cyclic encoding of preferred study time on a 24-hour clock
df["study_h"] = df["study_time"].map({"Morning":6,"Evening":18,"Late Night":24})
df["time_sin"] = np.sin(2*pi*df["study_h"]/24)
df["time_cos"] = np.cos(2*pi*df["study_h"]/24)

# Yes/No binary
yn = lambda s: s.map({"Yes":1,"No":0})
for c in ["pt_job","laptop","ECA","travel_impact","screen_sleep"]:
    df[c+"_b"] = yn(df[c])
df["Gender_b"] = df["Gender"].map({"Male":1,"Female":0})

# Multi-label phone use -> one-hot
phone = df["phone_use"].fillna("").str.get_dummies(sep=", ").astype(int)
phone.columns = [c.strip().replace(" ","_") for c in phone.columns]
for col in ["Study_Purpose","Social_Media","Entertainment","Gaming"]:
    if col not in phone.columns:
        phone[col] = 0
df = pd.concat([df, phone], axis=1)

# Transport mode for K-Means; raw category retained for K-Prototypes
df["trans_n"] = df["trans_mode"].map(
    {"University Bus":0,"Public Transport":1,"Private Vehicle":2})

log("Encoded variables created: CGPA, study_hrs_n, sleep_hrs_n, screen_n,")
log("travel_n, stress_n, fin_cond_n, balance_n, work_hrs_n, time_sin, time_cos,")
log("pt_job_b, laptop_b, ECA_b, travel_impact_b, screen_sleep_b, Gender_b,")
log("Study_Purpose, Social_Media, Entertainment, Gaming, trans_n")

# ----------------------------------------------------------------------------
# Step 3. Descriptive statistics
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 3 — DESCRIPTIVE STATISTICS")
log("=" * 78)

log(f"\nAge: M = {df['Age'].mean():.2f}, SD = {df['Age'].std():.2f}, "
    f"range = {df['Age'].min()}–{df['Age'].max()}")
log(f"\nGender:\n{df['Gender'].value_counts()}")
log(f"\nCGPA bands:\n{df['CGPA?'].value_counts()}")
log(f"\nStudy hours:\n{df['study_hrs'].value_counts()}")
log(f"\nTravel time:\n{df['travel_time'].value_counts()}")
log(f"\nStress:\n{df['stress'].value_counts()}")

# ----------------------------------------------------------------------------
# Step 4. Bivariate correlations with CGPA (with 95% CI)
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 4 — BIVARIATE CORRELATIONS WITH CGPA")
log("=" * 78)

def fisher_ci(r, n, alpha=0.05):
    """Confidence interval for Pearson r via Fisher z-transform."""
    if abs(r) >= 1: return r, r
    z, se = np.arctanh(r), 1/np.sqrt(n-3)
    zc = stats.norm.ppf(1-alpha/2)
    return np.tanh(z-zc*se), np.tanh(z+zc*se)

corr_vars = {
    "Academic self-rating": "acad_rating",
    "Daily study hours": "study_hrs_n",
    "Travel time": "travel_n",
    "Avoid screen before sleep": "screen_sleep_b",
    "Phone: study purpose": "Study_Purpose",
    "Time of study (morning bias)": "time_sin",
    "Phone: social media": "Social_Media",
    "Perceived stress": "stress_n",
    "Sleep duration": "sleep_hrs_n",
    "Daily screen time": "screen_n",
    "Extracurriculars": "ECA_b",
    "Part-time job": "pt_job_b",
    "Phone: entertainment": "Entertainment",
    "Weekly work hours": "work_hrs_n",
    "Travel impact (perceived)": "travel_impact_b",
    "Financial condition": "fin_cond_n",
    "Phone: gaming": "Gaming",
    "Gender (Male=1)": "Gender_b",
    "Age": "Age",
}

corr_rows = []
for label, col in corr_vars.items():
    s = df[[col,"CGPA"]].dropna()
    r, p = stats.pearsonr(s[col], s["CGPA"])
    lo, hi = fisher_ci(r, len(s))
    corr_rows.append({"variable":label,"col":col,"n":len(s),
                      "r":r,"ci_lo":lo,"ci_hi":hi,"p":p})
corr_df = pd.DataFrame(corr_rows).sort_values("r", key=abs, ascending=False)
log("\n" + corr_df[["variable","r","ci_lo","ci_hi","p"]].round(3).to_string(index=False))
corr_df.to_csv("correlations.csv", index=False)
log("\nSaved: correlations.csv")

# Forest plot of correlations
fig, ax = plt.subplots(figsize=(7,6))
plot_df = corr_df.iloc[::-1].reset_index(drop=True)
for i, row in plot_df.iterrows():
    color = "#2b6cb0" if row["r"] > 0 else "#c53030"
    ax.errorbar(row["r"], i, xerr=[[row["r"]-row["ci_lo"]],[row["ci_hi"]-row["r"]]],
                fmt="o", color=color, capsize=3, markersize=6)
    if row["p"] < 0.05:
        ax.text(row["ci_hi"]+0.02, i, "*", va="center", fontsize=14)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["variable"])
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Pearson correlation with CGPA (95% CI)")
ax.set_title("Bivariate correlates of CGPA (N = 200)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig01_correlations.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved figure: fig01_correlations.png")

# ----------------------------------------------------------------------------
# Step 5. Standardise features for clustering
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 5 — STANDARDISE FEATURES")
log("=" * 78)

CLUSTER_FEATS = [
    "acad_rating","study_hrs_n","screen_sleep_b","Study_Purpose","trans_n",
    "time_sin","time_cos","Entertainment","fin_cond_n","sleep_hrs_n",
    "travel_n","stress_n","Social_Media","ECA_b","work_hrs_n",
]
X = df[CLUSTER_FEATS].values
Xs = StandardScaler().fit_transform(X)
y_cgpa = df["CGPA"].values
log(f"Feature matrix X: shape {Xs.shape}, all standardised to zero mean, unit variance.")

# ----------------------------------------------------------------------------
# Step 6. Choose number of clusters
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 6 — CHOOSE NUMBER OF CLUSTERS")
log("=" * 78)

ks = list(range(2, 9))
diag_rows = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RNG, n_init=10).fit(Xs)
    sil = silhouette_score(Xs, km.labels_)
    ch = calinski_harabasz_score(Xs, km.labels_)
    db = davies_bouldin_score(Xs, km.labels_)
    diag_rows.append({"k":k,"inertia":km.inertia_,"silhouette":sil,
                      "calinski_harabasz":ch,"davies_bouldin":db})
diag_df = pd.DataFrame(diag_rows)
log("\n" + diag_df.round(3).to_string(index=False))

# Figure: elbow + silhouette
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(diag_df["k"], diag_df["inertia"], "o-", color="#2b6cb0")
axes[0].set(xlabel="k", ylabel="Within-cluster sum of squares", title="Elbow plot")
axes[0].axvline(3, color="red", ls="--", alpha=0.5)
axes[1].plot(diag_df["k"], diag_df["silhouette"], "o-", color="#dd6b20", label="Silhouette")
axes[1].set(xlabel="k", ylabel="Silhouette coefficient",
            title="Silhouette coefficient")
axes[1].axvline(3, color="red", ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig02_k_selection.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved figure: fig02_k_selection.png")
log("Selected k = 3 (silhouette + interpretability)")

K = 3

# ----------------------------------------------------------------------------
# Step 7. Fit clustering models
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 7 — FIT CLUSTERING MODELS")
log("=" * 78)

# Primary: K-Prototypes (mixed numeric + categorical) -- methodologically correct
try:
    from kmodes.kprototypes import KPrototypes
    NUM_COLS = ["acad_rating","study_hrs_n","sleep_hrs_n","screen_n",
                "travel_n","stress_n","work_hrs_n","fin_cond_n",
                "time_sin","time_cos"]
    CAT_COLS = ["trans_mode","pt_job","ECA","screen_sleep","Gender"]
    Xmix = df[NUM_COLS + CAT_COLS].copy()
    Xmix[NUM_COLS] = StandardScaler().fit_transform(Xmix[NUM_COLS])
    cat_idx = list(range(len(NUM_COLS), len(NUM_COLS)+len(CAT_COLS)))
    kp = KPrototypes(n_clusters=K, init="Cao", random_state=RNG, n_init=10, verbose=0)
    df["cluster_kproto"] = kp.fit_predict(Xmix.values, categorical=cat_idx)
    log("K-Prototypes fitted (primary).")
except ImportError:
    log("kmodes not installed. Falling back to K-Means as primary.")
    df["cluster_kproto"] = KMeans(n_clusters=K, random_state=RNG, n_init=10).fit_predict(Xs)

# Secondary: K-Means (continuous/Euclidean assumption)
df["cluster_kmeans"] = KMeans(n_clusters=K, random_state=RNG, n_init=10).fit_predict(Xs)
# Secondary: Hierarchical Ward (no centroid assumption)
df["cluster_ward"] = AgglomerativeClustering(n_clusters=K, linkage="ward").fit_predict(Xs)

log("\nCluster sizes:")
log(df["cluster_kproto"].value_counts().sort_index().to_string())

# Re-label clusters by ascending mean CGPA so Cluster 1 = lowest, Cluster 3 = highest.
def relabel_by_cgpa(labels, y):
    order = pd.Series(y).groupby(labels).mean().sort_values().index.tolist()
    mapping = {old: new+1 for new, old in enumerate(order)}
    return pd.Series(labels).map(mapping).values

df["cluster"] = relabel_by_cgpa(df["cluster_kproto"], df["CGPA"])
log("\nClusters re-labelled by ascending mean CGPA (1 = lowest, 3 = highest).")

# ----------------------------------------------------------------------------
# Step 8. Profile and label the clusters
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 8 — PROFILE THE CLUSTERS")
log("=" * 78)

profile_feats = ["acad_rating","study_hrs_n","sleep_hrs_n","screen_n",
                 "travel_n","stress_n","work_hrs_n","fin_cond_n",
                 "Study_Purpose","Social_Media","Entertainment","ECA_b",
                 "screen_sleep_b","CGPA"]
profile = df.groupby("cluster")[profile_feats].mean().round(2)
profile["n"] = df["cluster"].value_counts().sort_index()
log("\n" + profile.T.to_string())
profile.to_csv("cluster_profile.csv")
log("\nSaved: cluster_profile.csv")

# ----------------------------------------------------------------------------
# Step 9. ANOVA + Tukey HSD: cluster differences in CGPA
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 9 — ANOVA + TUKEY HSD: CLUSTER DIFFERENCES IN CGPA")
log("=" * 78)

groups = [df.loc[df["cluster"]==c, "CGPA"].values for c in sorted(df["cluster"].unique())]
F, p = stats.f_oneway(*groups)
grand = df["CGPA"].mean()
ss_between = sum(len(g)*(g.mean()-grand)**2 for g in groups)
ss_total = sum((df["CGPA"]-grand)**2)
eta2 = ss_between/ss_total
log(f"\nOne-way ANOVA: F({K-1}, {len(df)-K}) = {F:.3f}, p = {p:.4f}, eta^2 = {eta2:.3f}")

# Tukey HSD post-hoc
tukey = pairwise_tukeyhsd(df["CGPA"], df["cluster"], alpha=0.05)
log("\nTukey HSD pairwise comparisons:")
log(str(tukey))

# Kruskal-Wallis robustness
H, pkw = stats.kruskal(*groups)
log(f"\nKruskal-Wallis (non-parametric robustness): H = {H:.3f}, p = {pkw:.4f}")

# Boxplot
palette = {1:"#c53030", 2:"#dd6b20", 3:"#2b6cb0"}
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.boxplot(data=df, x="cluster", y="CGPA", hue="cluster",
            palette=palette, ax=ax, legend=False, width=0.5)
sns.stripplot(data=df, x="cluster", y="CGPA", color="black",
              alpha=0.35, size=3, ax=ax)
ax.set(xlabel="Cluster (1 = lowest CGPA, 3 = highest)",
       ylabel="CGPA (band midpoint)",
       title=f"CGPA by cluster — F({K-1},{len(df)-K})={F:.2f}, p<.001, eta2={eta2:.3f}")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig03_cgpa_by_cluster.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved figure: fig03_cgpa_by_cluster.png")

# Robustness of cluster-CGPA ANOVA across algorithms
log("\nRobustness across clustering algorithms:")
for algo in ["cluster_kproto","cluster_kmeans","cluster_ward"]:
    g = [df.loc[df[algo]==c,"CGPA"].values for c in sorted(df[algo].unique())]
    Fa, pa = stats.f_oneway(*g)
    eta = sum(len(gi)*(gi.mean()-grand)**2 for gi in g)/ss_total
    log(f"  {algo:<18} F = {Fa:.2f}, p = {pa:.4f}, eta^2 = {eta:.3f}")

# Hierarchical dendrogram for context
fig, ax = plt.subplots(figsize=(13, 4))
Z = linkage(Xs, method="ward")
dendrogram(Z, ax=ax, leaf_font_size=6, no_labels=True,
           color_threshold=0.7*max(Z[:,2]))
ax.set(title="Ward hierarchical dendrogram (N = 200)",
       xlabel="Student", ylabel="Distance")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig04_dendrogram.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved figure: fig04_dendrogram.png")

# ----------------------------------------------------------------------------
# Step 10. MCA visualisation (replaces PCA for categorical data)
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 10 — MCA VISUALISATION")
log("=" * 78)

try:
    import prince
    cat_cols = ["CGPA?","study_hrs","study_time","sleep_hrs","screen_time",
                "travel_time","stress","fin_cond","trans_mode","balance_diff",
                "pt_job","laptop","ECA","travel_impact","screen_sleep","Gender"]
    mca_df = df[cat_cols].astype(str)
    mca = prince.MCA(n_components=2, random_state=RNG).fit(mca_df)
    coords = mca.transform(mca_df).values

    fig, ax = plt.subplots(figsize=(7,5.5))
    for c in sorted(df["cluster"].unique()):
        m = df["cluster"] == c
        ax.scatter(coords[m,0], coords[m,1], s=55, alpha=0.75, color=palette[c],
                   edgecolor="white", linewidth=0.6, label=f"Cluster {c} (n={m.sum()})")
    ax.set(xlabel="MCA Dim 1", ylabel="MCA Dim 2",
           title="Student clusters in MCA space (categorical-data analogue of PCA)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTDIR}/fig05_mca_clusters.png", dpi=200, bbox_inches="tight")
    plt.close()
    log("Saved figure: fig05_mca_clusters.png")
except ImportError:
    # Fall back to PCA if prince not installed
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2, random_state=RNG)
    coords = pca.fit_transform(Xs)
    log("prince not installed; using PCA fallback.")
    fig, ax = plt.subplots(figsize=(7,5.5))
    for c in sorted(df["cluster"].unique()):
        m = df["cluster"] == c
        ax.scatter(coords[m,0], coords[m,1], s=55, alpha=0.75, color=palette[c],
                   edgecolor="white", linewidth=0.6, label=f"Cluster {c} (n={m.sum()})")
    ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
           ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
           title="Student clusters in PCA space (fallback)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTDIR}/fig05_pca_clusters.png", dpi=200, bbox_inches="tight")
    plt.close()
    log("Saved figure: fig05_pca_clusters.png")


# ============================================================================
# PART B — PREDICTIVE ML MODEL
# ============================================================================

# ----------------------------------------------------------------------------
# Step 11. Build the predictor matrix
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 11 — BUILD PREDICTOR MATRIX FOR ML MODEL")
log("=" * 78)

PRED_FEATS = [
    "acad_rating","study_hrs_n","sleep_hrs_n","screen_n","travel_n",
    "stress_n","fin_cond_n","work_hrs_n","time_sin","time_cos",
    "Study_Purpose","Social_Media","Entertainment","Gaming",
    "ECA_b","screen_sleep_b","pt_job_b","laptop_b","travel_impact_b",
    "Gender_b","Age","trans_n","balance_n",
]
Xp_raw = df[PRED_FEATS].values
yp = df["CGPA"].values
log(f"Predictor matrix: {Xp_raw.shape}, target: CGPA (continuous, midpoint of band).")
log(f"Predictors: {PRED_FEATS}")

# ----------------------------------------------------------------------------
# Step 12. Train/test split + cross-validation setup
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 12 — TRAIN / TEST SPLIT (80/20)")
log("=" * 78)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    Xp_raw, yp, test_size=0.20, random_state=RNG)
scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)
log(f"Train: {X_train.shape}, Test: {X_test.shape}")

cv = KFold(n_splits=5, shuffle=True, random_state=RNG)

# ----------------------------------------------------------------------------
# Step 13. Fit and compare candidate models
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 13 — COMPARE CANDIDATE MODELS (5-fold CV on training set)")
log("=" * 78)

models = {
    "OLS"            : LinearRegression(),
    "Ridge"          : Ridge(alpha=1.0, random_state=RNG),
    "Lasso"          : Lasso(alpha=0.05, random_state=RNG, max_iter=20000),
    "LassoCV"        : LassoCV(cv=cv, random_state=RNG, max_iter=20000),
    "RandomForest"   : RandomForestRegressor(n_estimators=500, max_depth=6,
                                             random_state=RNG, n_jobs=-1),
    "GradBoost"      : GradientBoostingRegressor(n_estimators=300, max_depth=3,
                                                 learning_rate=0.05,
                                                 random_state=RNG),
}

cv_rows = []
for name, m in models.items():
    r2  = cross_val_score(m, X_train, y_train, cv=cv, scoring="r2")
    mae = -cross_val_score(m, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    cv_rows.append({"model":name,"r2_mean":r2.mean(),"r2_sd":r2.std(),
                    "mae_mean":mae.mean(),"mae_sd":mae.std()})
cv_df = pd.DataFrame(cv_rows).sort_values("r2_mean", ascending=False)
log("\n" + cv_df.round(3).to_string(index=False))

best_name = cv_df.iloc[0]["model"]
log(f"\nBest cross-validated model: {best_name}")

# ----------------------------------------------------------------------------
# Step 14. Hyperparameter tuning of the winning model
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 14 — HYPERPARAMETER TUNING")
log("=" * 78)

if best_name in ("Lasso","LassoCV"):
    grid = {"alpha":[0.001,0.005,0.01,0.02,0.05,0.1,0.2]}
    base = Lasso(random_state=RNG, max_iter=20000)
elif best_name == "Ridge":
    grid = {"alpha":[0.01,0.1,1.0,10,100]}
    base = Ridge(random_state=RNG)
elif best_name == "RandomForest":
    grid = {"n_estimators":[300,500,800],"max_depth":[3,5,7,None],
            "min_samples_leaf":[1,2,5]}
    base = RandomForestRegressor(random_state=RNG, n_jobs=-1)
elif best_name == "GradBoost":
    grid = {"n_estimators":[200,400,600],"max_depth":[2,3,4],
            "learning_rate":[0.02,0.05,0.1]}
    base = GradientBoostingRegressor(random_state=RNG)
else:
    grid = {}
    base = models[best_name]

if grid:
    gs = GridSearchCV(base, grid, cv=cv, scoring="r2", n_jobs=-1)
    gs.fit(X_train, y_train)
    log(f"Best params: {gs.best_params_}")
    log(f"Best CV R^2: {gs.best_score_:.3f}")
    final_model = gs.best_estimator_
else:
    final_model = base.fit(X_train, y_train)
    log("No tunable hyperparameters; using default.")

# ----------------------------------------------------------------------------
# Step 15. Final evaluation on held-out test set
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 15 — FINAL EVALUATION ON HELD-OUT TEST SET")
log("=" * 78)

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
test_r2  = r2_score(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(np.mean((y_test - y_pred)**2))
log(f"Test R^2  = {test_r2:.3f}")
log(f"Test MAE  = {test_mae:.3f} CGPA points")
log(f"Test RMSE = {test_rmse:.3f} CGPA points")

# Predicted vs actual plot
fig, ax = plt.subplots(figsize=(5.5,5.5))
ax.scatter(y_test, y_pred, s=55, alpha=0.75, color="#2b6cb0",
           edgecolor="white", linewidth=0.6)
mn, mx = min(y_test.min(), y_pred.min())-0.2, max(y_test.max(), y_pred.max())+0.2
ax.plot([mn,mx],[mn,mx], "r--", alpha=0.6, label="y = x")
ax.set(xlabel="Actual CGPA", ylabel="Predicted CGPA",
       xlim=(mn,mx), ylim=(mn,mx),
       title=f"Predicted vs actual CGPA (test R^2 = {test_r2:.2f})")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig06_pred_vs_actual.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved figure: fig06_pred_vs_actual.png")

# ----------------------------------------------------------------------------
# Step 16. Feature importance analysis
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 16 — FEATURE IMPORTANCE")
log("=" * 78)

# Use Random Forest (always available) for an importance ranking on training set
rf = RandomForestRegressor(n_estimators=500, max_depth=6,
                           random_state=RNG, n_jobs=-1).fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=PRED_FEATS).sort_values()

fig, ax = plt.subplots(figsize=(7,7))
imp.plot(kind="barh", ax=ax, color="#2b6cb0")
ax.set(xlabel="Importance", title="Feature importance (Random Forest, train set)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig07_feature_importance.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved figure: fig07_feature_importance.png")

log("\nTop 10 features by Random Forest importance:")
log(imp.iloc[::-1].head(10).round(3).to_string())

# Lasso coefficients as a parametric counterpart
lcv = LassoCV(cv=cv, random_state=RNG, max_iter=20000).fit(X_train, y_train)
coef = pd.Series(lcv.coef_, index=PRED_FEATS)
nz = coef[coef != 0].sort_values(key=abs, ascending=False)
log(f"\nLassoCV optimal alpha: {lcv.alpha_:.4f}")
log(f"Non-zero coefficients ({len(nz)}/{len(coef)}):")
log(nz.round(3).to_string())

# ----------------------------------------------------------------------------
# Step 17. Save everything
# ----------------------------------------------------------------------------
log("\n" + "=" * 78)
log("STEP 17 — SAVE OUTPUTS")
log("=" * 78)

# Save the final dataset with cluster labels for downstream use
out_cols = ["Age","Gender","CGPA?","CGPA","study_hrs","study_time","acad_rating",
            "fin_cond","pt_job","work_hrs","screen_time","phone_use","travel_time",
            "trans_mode","sleep_hrs","ECA","stress","screen_sleep","balance_diff",
            "cluster"]
df[out_cols].to_csv("dataset_with_clusters.csv", index=False)
log("Saved: dataset_with_clusters.csv (original data + cluster labels)")

# Save full results summary
with open("results_summary.txt", "w") as f:
    f.write("\n".join(LOG_LINES))
log("Saved: results_summary.txt (full console output)")

# Persist key numeric results as JSON for the paper
results = {
    "n": int(len(df)),
    "anova": {"F": float(F), "p": float(p), "eta_squared": float(eta2)},
    "kruskal": {"H": float(H), "p": float(pkw)},
    "cluster_means": {int(c): float(df.loc[df["cluster"]==c, "CGPA"].mean())
                      for c in sorted(df["cluster"].unique())},
    "cluster_sizes": {int(c): int((df["cluster"]==c).sum())
                      for c in sorted(df["cluster"].unique())},
    "best_model": str(best_name),
    "test_r2": float(test_r2),
    "test_mae": float(test_mae),
    "test_rmse": float(test_rmse),
    "lasso_alpha": float(lcv.alpha_),
    "lasso_nonzero_features": list(nz.index),
}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
log("Saved: results.json (machine-readable headline numbers)")

log("\n" + "=" * 78)
log("PIPELINE COMPLETE")
log("=" * 78)
log(f"\nFigures in: {OUTDIR}/")
log("  fig01_correlations.png        Bivariate correlations forest plot")
log("  fig02_k_selection.png         Elbow + silhouette diagnostics")
log("  fig03_cgpa_by_cluster.png     Boxplot of CGPA by cluster")
log("  fig04_dendrogram.png          Hierarchical dendrogram")
log("  fig05_mca_clusters.png        MCA 2-D cluster scatter")
log("  fig06_pred_vs_actual.png      Test-set predicted vs actual CGPA")
log("  fig07_feature_importance.png  Random Forest feature importances")
log("\nTables / data files:")
log("  correlations.csv              Full Pearson correlation table")
log("  cluster_profile.csv           Cluster-mean profiles")
log("  dataset_with_clusters.csv     Data with cluster labels appended")
log("  results.json                  Machine-readable headline numbers")
log("  results_summary.txt           Full text log")

# Re-write the log one more time so that the final lines are included
with open("results_summary.txt", "w") as f:
    f.write("\n".join(LOG_LINES))

STEP 1 — LOAD AND CLEAN THE DATA
Loaded 200 rows, 21 columns
Columns: ['Age', 'Gender', 'CGPA?', 'study_hrs', 'study_time', 'acad_rating', 'fin_cond', 'pt_job', 'work_hrs', 'smartphone', 'laptop', 'screen_time', 'phone_use', 'travel_time', 'trans_mode', 'sleep_hrs', 'ECA', 'travel_impact', 'stress', 'screen_sleep', 'balance_diff']
Missing values:
work_hrs    121
dtype: int64

STEP 2 — ENCODE VARIABLES (ordinal / cyclic / binary / one-hot)
Encoded variables created: CGPA, study_hrs_n, sleep_hrs_n, screen_n,
travel_n, stress_n, fin_cond_n, balance_n, work_hrs_n, time_sin, time_cos,
pt_job_b, laptop_b, ECA_b, travel_impact_b, screen_sleep_b, Gender_b,
Study_Purpose, Social_Media, Entertainment, Gaming, trans_n

STEP 3 — DESCRIPTIVE STATISTICS

Age: M = 22.21, SD = 1.40, range = 19–25

Gender:
Gender
Male      177
Female     23
Name: count, dtype: int64

CGPA bands:
CGPA?
3.5-4.0    69
3.0-3.5    68
2.5-3.0    53
2.0-2.5    10
Name: count, dtype: int64

Study hours:
study_hrs
1-2 hours    